# Named Entity Recognition (NER) with GLiNER2

To use this notebook on relation extraction, especially notebook nr. 4 on rule-based relation extraction, you need data that has been annotated with named entities. If this is not the case, you can use this notebook to annotate your data using the GLiNER2 model.

For this example, we will use a pre-trained GLiNER2 model that has been trained to extract named entities from Latin hagiographic texts. To learn more about how to train your own GLiNER2 models and adapters, you can consult the GLiNER2 documentation [https://github.com/fastino-ai/GLiNER2 ].

In [ ]:
# run this if GLiNER2 is not yet installed in your environment
!pip install gliner2

## GLiNER2 Models Comparison
You can choose from a variety of GLiNER2 models depending on your language and accuracy requirements. Below is a comparison of some popular GLiNER2 models:


| Model | Languages          | Best For                    |
|-------|--------------------|-----------------------------|
| `fastino/gliner2-base-v1` | English            | General English NER         |
| `fastino/gliner2-large-v1` | EN, FR, SP         | High-accuracy English NER   |
| `fastino/gliner2-multi-v1` | 12+ languages      | Multilingual projects       |
| `fastino/gliner2-multi-pii-v1` | EN, FR, SP, DE, IT | Multilingual  PII detection |
| `fastino/gliner2-spanish-v1` | Spanish            | Spanish NER                 |
| `fastino/gliner2-french-v1` | French             | French NER                  |
| `GhentCDH/Latin-Hagiography-NER` | Latin              | Latin hagiographic texts    |


**Note:** as the project for which this repo is crated focuses on Latin hagiographic texts, all the examples in this repository will use Latin examples. We will use the "GhentCDH/Latin-Hagiography-NER" model. However, you can choose any other model that is trained on the language of your documents.


## GLiNER2 entity extraction
With this codeblock, you can extract named entities from your text using the GLiNER2 model. The output will be a JSON file containing the extracted entities, their spans, and confidence scores. The output will be saved in the same directory as your input text file, with the suffix "_NER.json" added to the base name of your input file.

If you don't have your own text file, you can use the sample text file provided in this repository: ```sample_texts/rule-based-example.txt```. You can also use your own text file by changing the path in the code below.

In [3]:
#we use our pretrained "GhentCDH/Latin-Hagiography-NER" to detect entities in our text, chose a model that is trained on the language of your documents

#this

from gliner2 import GLiNER2
import json
import os
from gliner2 import GLiNER2
from gliner_to_labelstudio import (
    load_gliner_schema_config,
    create_gliner_schema_from_config_file,
)

SCHEMA_CONFIG_PATH = "./gliner_schema_hagiographics_NER.json"
schema_config = load_gliner_schema_config(SCHEMA_CONFIG_PATH)

model = GLiNER2.from_pretrained("GhentCDH/Latin-Hagiography-NER")

extractor = model

schema = create_gliner_schema_from_config_file(extractor, SCHEMA_CONFIG_PATH)

input_file = "sample_texts/rule-based-example.txt"  # change this to your text file path
base_name = os.path.splitext(input_file)[0]  # removes ".txt" -> "located_at_text"

output_file = f"{base_name}_NER.json"

with open(input_file, "r", encoding="utf-8") as f:
    text = f.read()

results = results = extractor.extract(text, schema, threshold=0.1, include_confidence=True, include_spans=True,
                                      format_results=False)

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(json.dumps(results, indent=2, ensure_ascii=False))

[transformers] You are using a model of type `extractor` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


🧠 Model Configuration
Encoder model      : microsoft/mdeberta-v3-base
Counting layer     : count_lstm
Token pooling      : first
{
  "entities": [
    {
      "person": [
        {
          "text": "Sanctus Amandulus",
          "confidence": 0.9777584075927734,
          "start": 0,
          "end": 17
        },
        {
          "text": "Vir sanctus",
          "confidence": 0.9291267395019531,
          "start": 178,
          "end": 189
        }
      ],
      "group": [
        {
          "text": "fratres",
          "confidence": 0.969921886920929,
          "start": 212,
          "end": 219
        },
        {
          "text": "peregrinis",
          "confidence": 0.9499704837799072,
          "start": 660,
          "end": 670
        },
        {
          "text": "Multi aegroti",
          "confidence": 0.8130289316177368,
          "start": 392,
          "end": 405
        }
      ],
      "object": [
        {
          "text": "arca",
          "confidence": 0.89

## text normalization

[this step is only important if your text is in Latin and you will be using LatinCy for the rule-based relation extraction in notebook 4]

The LatinCy pipeline uses a normalizer to convert the orthography of the text to standardised form. Converting your text to this form before doing anything else is essential for the pipeline to work properly. As Æ and Œ ligatures are automatically converted to AE and OE. This would be problematic as the span of a word changes and the entities would not be aligned properly.

The following code will convert your text to the standardised form and change the entity span. There is also the possibility withing the code to remove any bracketed text, which is often used to indicate editorial additions or corrections. However, it is important to note that this may also remove important information from the text, so use this option with caution.

In [4]:
import re
import json

LIGATURE_MAP = {
    "v": "u", "V": "U",
    "j": "i", "J": "I",
    "æ": "ae", "Æ": "Ae",  # works mid-word too — it's a per-character map
    "œ": "oe", "Œ": "Oe",
}


def find_bracket_spans(text, pattern=r"\[Col\.[^\]]*\]"):
    """Editorial apparatus insertions like [Col. 0205E], number-only brackets excluded."""
    return [(m.start(), m.end()) for m in re.finditer(pattern, text)]


def normalize_text_and_build_map(text, strip_brackets=True, collapse_whitespace=True):
    remove_mask = [False] * len(text)
    if strip_brackets:
        for s, e in find_bracket_spans(text):
            for i in range(s, e):
                remove_mask[i] = True

    out_chars = []
    offset_map = []
    running_len = 0
    prev_emitted_is_space = True  # eats a leading space left behind by a removed bracket

    for i, ch in enumerate(text):
        offset_map.append(running_len)
        replacement = "" if remove_mask[i] else LIGATURE_MAP.get(ch, ch)

        if collapse_whitespace and replacement == " " and prev_emitted_is_space:
            replacement = ""

        if replacement:
            out_chars.append(replacement)
            running_len += len(replacement)
            prev_emitted_is_space = replacement[-1] == " "

    offset_map.append(running_len)  # end-of-string sentinel
    normalized_text = "".join(out_chars)
    return normalized_text, offset_map

In [5]:
def remap_entities(entities_block, offset_map, normalized_text):
    problems, new_block = [], []
    for group in entities_block:
        new_group = {}
        for label, mentions in group.items():
            new_mentions = []
            for m in mentions:
                start, end = m["start"], m["end"]
                new_start, new_end = offset_map[start], offset_map[end]
                if new_end <= new_start and end > start:
                    problems.append((label, m, "entity collapsed — was inside removed bracket text"))
                    continue
                new_m = dict(m)
                new_m.update(start=new_start, end=new_end,
                             text=normalized_text[new_start:new_end],
                             orig_start=start, orig_end=end, orig_text=m.get("text"))
                new_mentions.append(new_m)
            new_group[label] = new_mentions
        new_block.append(new_group)
    return new_block, problems

In [8]:
text = open("sample_texts/rule-based-example.txt", encoding="utf-8").read() # <--- change this path to your text
base_name = os.path.splitext(input_file)[0]

NER = json.load(open(f"{base_name}_NER.json", encoding="utf-8"))

norm_text, offset_map = normalize_text_and_build_map(text)
new_entities, problems = remap_entities(NER["entities"], offset_map, norm_text)

print("Brackets removed:", find_bracket_spans(text))
print(f"Remapped OK: {sum(len(v) for g in new_entities for v in g.values())} mentions")
if problems:
    print(f"{len(problems)} problem(s):", problems)

with open(f"{base_name}_norm.txt", "w", encoding="utf-8") as f:
    f.write(norm_text)

with open(f"{base_name}_NER_norm.json", "w", encoding="utf-8") as f:
    json.dump({"entities": new_entities}, f, ensure_ascii=False, indent=2)


Brackets removed: []
Remapped OK: 28 mentions


# Spacy convert

The rule-based relation extraction pipeline in notebook nr. 4 uses Spacy to process the text. Therefore, we need to convert the GLiNER2 output to a Spacy compatible format. The following code will do this for you.

In [9]:
# convert GLiNER NER output (separate text + NER files) to spaCy training format
# Output shape matches ls_task_to_spacy_record: {"text", "entities", "relations"}

import json
import uuid
from pathlib import Path

# --- Config (self-contained) ---
TEXT_PATH = Path("sample_texts/rule-based-example_norm.txt")  # <-- change the path

# Text files are named "<base>_norm.txt"; NER files are named "<base>_NER_norm.json".
# Derive the NER/output paths from TEXT_PATH so you only edit one line above.
text_stem = TEXT_PATH.stem
base_name = text_stem[:-5] if text_stem.endswith("_norm") else text_stem  # strip trailing "_norm"
suffix = "_norm" if text_stem.endswith("_norm") else ""

NER_PATH = TEXT_PATH.with_name(f"{base_name}_NER{suffix}.json")
SPACY_OUT_PATH = TEXT_PATH.with_name(f"{base_name}_NER{suffix}_spacy.json")

# --- Safety checks ---
if not TEXT_PATH.exists():
    raise FileNotFoundError(f"Missing text file: {TEXT_PATH}")
if not NER_PATH.exists():
    raise FileNotFoundError(f"Missing NER file: {NER_PATH}")

# --- Load inputs ---
text = TEXT_PATH.read_text(encoding="utf-8")
with NER_PATH.open("r", encoding="utf-8") as f:
    gliner_results = json.load(f)


def gliner_to_spacy_record(gliner_json, text):
    """
    Convert GLiNER output into the same spaCy record shape used for the
    Label Studio pipeline:
        {"text": ..., "entities": [{"start","end","label","text","id"}, ...], "relations": []}
    GLiNER doesn't produce relations, so "relations" is always empty here --
    the key exists purely so downstream code (built for the LS side) can
    treat both sources identically without special-casing.
    """
    entities = []
    for entity_group in gliner_json.get("entities", []):
        for label, mentions in entity_group.items():
            for mention in mentions:
                start = mention["start"]
                end = mention["end"]
                entities.append({
                    "start": start,
                    "end": end,
                    "label": label,
                    "text": mention.get("text", text[start:end]),
                    "id": uuid.uuid4().hex[:10],  # matches the LS id style
                })

    entities.sort(key=lambda e: e["start"])

    return {"text": text, "entities": entities, "relations": []}


record = gliner_to_spacy_record(gliner_results, text)

with SPACY_OUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(record, f, indent=2, ensure_ascii=False)

print(f"Saved: {SPACY_OUT_PATH}")
print(json.dumps(record, indent=2, ensure_ascii=False))

Saved: sample_texts\rule-based-example_NER_norm_spacy.json
{
  "text": "Sanctus Amandulus, genere humili natus, in monasterio apud flumen educatus est. Ibi erat ecclesia in colle sita, cui uicinum erat xenodochium prope portam ciuitatis constitutum. Uir sanctus saepe morabatur inter fratres, sed etiam ad speluncam sub monte positam se recipiebat ad orandum. Postea oratorium condidit ad radices siluae, ubi fons ante aedem manabat et crux supra altare eminebat. Multi aegroti conueniebant ad eum, quia fama sanctitatis eius per totam regionem diffusa erat. Tandem sepultus est in basilica intra muros urbis, ubi etiam nunc arca reliquiarum eius iuxta maius altare seruari dicitur, monasterium autem e regione fori adhuc uisitur a peregrinis.",
  "entities": [
    {
      "start": 0,
      "end": 17,
      "label": "person",
      "text": "Sanctus Amandulus",
      "id": "5bf3fffd3d"
    },
    {
      "start": 43,
      "end": 53,
      "label": "institution",
      "text": "monasterio",
     

# Next Steps

Perfect! You have successfully annotated your text with named entities using the GLiNER2 model, normalized the text for Latin, and converted the output to a spaCy-compatible format. Now you are ready to do rule-based relation extraction in notebook nr. 4. Make sure to use the normalized text and the spaCy-compatible NER output throughout this notebook.